# C11-neural-training — Practice p14 — Solution


**Type:** integrative (parts consume earlier results) · **Difficulty:** core · **Concepts:** softmax, cross-entropy-loss


The same shifted logits drive probabilities, stable log-sum-exp loss, and the
fused batch-mean gradient.


In [ ]:
import numpy as np

def softmax_ce_with_gradient(logits, labels):
    z=np.asarray(logits,dtype=np.float64); y=np.asarray(labels)
    if z.ndim!=2 or y.ndim!=1 or z.shape[0]!=y.shape[0] or z.shape[0]==0 or z.shape[1]==0:
        raise ValueError("invalid dimensions")
    if not np.issubdtype(y.dtype,np.integer) or np.any(y<0) or np.any(y>=z.shape[1]):
        raise ValueError("invalid labels")
    shifted=z-z.max(axis=1,keepdims=True); exp_shifted=np.exp(shifted); sums=exp_shifted.sum(axis=1,keepdims=True)
    probabilities=exp_shifted/sums
    loss=float(np.mean(np.log(sums[:,0])-shifted[np.arange(z.shape[0]),y]))
    gradient=probabilities.copy(); gradient[np.arange(z.shape[0]),y]-=1.0; gradient/=z.shape[0]
    return {"probabilities":probabilities,"loss":loss,"gradient":gradient}

Z_p14=np.array([[10000.,9998.,-10000.],[-9999.,-10000.,-10001.]],dtype=np.float64); y_p14=np.array([0,2])
result_p14=softmax_ce_with_gradient(Z_p14,y_p14)


### Answer check


In [ ]:
assert np.all(np.isfinite(result_p14["probabilities"])) and np.isfinite(result_p14["loss"])
assert np.allclose(result_p14["probabilities"].sum(1),1,atol=1e-11,rtol=1e-9)
assert np.allclose(result_p14["gradient"].sum(1),0,atol=1e-11,rtol=1e-9)
for i,j in [(0,0),(1,2)]:
    delta=np.zeros_like(Z_p14); delta[i,j]=1e-6
    numeric=(softmax_ce_with_gradient(Z_p14+delta,y_p14)["loss"]-softmax_ce_with_gradient(Z_p14-delta,y_p14)["loss"])/(2e-6)
    assert np.isclose(numeric,result_p14["gradient"][i,j],atol=2e-6,rtol=2e-5)
